In [11]:
import os, sys
from pathlib import Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
import numpy as np
import pandas as pd
import functions1 as f1  
import functions2 as f2
import importlib
import config
import risk_matrix
import books

importlib.reload(books)
importlib.reload(risk_matrix)
importlib.reload(f2)

print('params:\n', config.params)
window_start="2025-1-02"
holdings=books.IBKR_live
ohlc = False

rets_df, prices_df,  weights =risk_matrix.build_returns_matrix_in_chf(
                holdings,
                params=config.params,
                window_start=window_start)

print('prices_df\n', rets_df.head(1))

results = risk_matrix.portfolio_risk(rets_df, weights)

pf_vol = results["port_vol"]
summary_all = results["summary"]
summary = summary_all[~summary_all.index.str.startswith("CASH_")].copy()

print('summary', summary)
assets = summary.index

weights = summary["Weight"].astype(float)
mrc = summary["MRC"].astype(float)

desired_dd = 0.04

ohlc = True
fx_map = f2.make_fx_map(holdings, config.params, usd_shift=True, ohlc=ohlc)
high_df, low_df, close_df, close_local_df = f2.base_ccy_assets_px_df(holdings, fx_map, config.params, ohlc)


# ohlc_by_asset: dict like {asset: df_with_cols['High','Low','Close']}
# If your variable is named differently, replace 'ohlc_by_asset' below.

# Ensure 'w' exists (you used it below)
w = weights
# remove cash from weights
w=weights[~pd.Index(assets).str.startswith("CASH_")]
print('type assets:', type(assets))
print('------len weights:', len(w), 'len assets:', len(assets))
# 1) Compute ATR% (Wilder) from OHLC in the SAME currency basis as weights/returns (CHF)
# Expect DataFrames: high_df, low_df, close_df with columns matching 'assets'
# If you have them under different names, adjust below.
high = high_df[assets]
low = low_df[assets]
close = close_df[assets]
# print('close_df\n', close_df.head())
# print('close\n', close.head())

prev_close = close.shift(1)
tr = pd.concat([
    (high - low),
    (high - prev_close).abs(),
    (low - prev_close).abs()
], axis=1).groupby(level=0, axis=1).max()  # max of the three TR components per asset

atr = tr.ewm(alpha=1/14, adjust=False).mean()       # Wilder's 14-period RMA
atr_pct = (atr.iloc[-1] / close.iloc[-1]).astype(float).clip(lower=1e-6)

# 2) Per-asset DD budgets by absolute TRC (or use equal share)
trc_abs = (w * mrc).abs()
share = trc_abs / trc_abs.sum()
b = desired_dd * share                                # sum(b) = desired_dd

# 3) L1 stop distances and ATR multiples
abs_w = w.abs().clip(lower=1e-12)


# Map categories: default tactical; override specific tickers
cat = {  # examples; edit to your book
    "SGLN": "diversifier",
    "SILG": "diversifier",
    # "YCA": "diversifier",   # set if you want it wider or excluded
}
cat_s = pd.Series({a: cat.get(a, "tactical") for a in assets})

min_k_by_cat = {"core": 4.0, "tactical": 1.5, "diversifier": 3.5, "hedge": float("inf")}
include_in_stops = {"core": True, "tactical": True, "diversifier": True, "hedge": False}

min_k = cat_s.map(min_k_by_cat).astype(float)
take_stop = cat_s.map(include_in_stops).fillna(True).astype(bool)

min_pct, max_pct = 0.002, 0.10

# Initial L1 proposal
s = (b / abs_w).astype(float)

# Apply category floors and inclusion mask
floor = np.maximum(min_pct, (min_k * atr_pct).astype(float))
s[~take_stop] = np.nan
s[take_stop] = np.minimum(np.maximum(s[take_stop], floor[take_stop]), max_pct)

# Rescale down on included names to hit desired_dd exactly
L1 = float((abs_w[take_stop] * s[take_stop]).sum())
if L1 > desired_dd:
    s.loc[take_stop] *= desired_dd / L1

stop_pct = s.astype(float)             # optional floors/ceilings on distance (0.2%–10%)
k_multiple = (stop_pct / atr_pct)     # optional caps on ATR multiples

# 4) Stop prices
last_px_local = close_local_df[assets].iloc[-1].astype(float)

stop_px_local = pd.Series(
    np.where(w >= 0,
             last_px_local * (1.0 - stop_pct),
             last_px_local * (1.0 + stop_pct)),
    index=assets, name="stop_price_local"
)
last_px_chf = close.iloc[-1].astype(float).reindex(assets)
stop_px_chf = pd.Series(
    np.where(w >= 0, last_px_chf * (1.0 - stop_pct), last_px_chf * (1.0 + stop_pct)),
    index=assets, name="stop_price"
)

# 5)  L1 sum check still in CHF basis: worst-case sum should hit target DD.
dd_check = float((abs_w[take_stop] * stop_pct[take_stop]).sum())
print(f"L1 sum check ~ {dd_check:.4f} (target {desired_dd:.4f})")

out = pd.DataFrame({
    "Weight": w,
    "ATR_pct": atr_pct,
    "stop_pct": stop_pct,
    "k_ATR_mult": k_multiple,
    "last_px_local": last_px_local,
    "stop_px_local": stop_px_local,
    "last_px_chf": last_px_chf,
    "stop_px_chf": stop_px_chf
}).round(6)

print("Portfolio σ (annualized, CHF): ", pf_vol)

# Optionally drop cash/zero weights
# mask_live = (w.abs() > 1e-6) & ~pd.Index(assets).str.startswith("CASH_")
# print(out[mask_live])
print(out)

params:
 {'from': '2019-5-16', 'to': '2025-10-18', 'api_token': '68b5e3ff8bb5d2.47603652', 'max_age:': 12, 'url': 'https://eodhd.com/api/eod/'}
note: adjusted_close != close; using adjusted_close
note: adjusted_close != close; using adjusted_close
note: adjusted_close != close; using adjusted_close
note: adjusted_close != close; using adjusted_close
After alignment only 139 rows remain (expected 289 days 00:00:00). Data source may not have full history.
LOOKBACK DAYS/REGIME: 2025-01-02 00:00:00 to 2025-10-18 00:00:00  (289 days)
Total portfolio value (CHF): 24976.38
prices_df
                  VEU      EMIM      VUAG      ISJP      IEMS      XXSC  \
Date                                                                     
2025-01-06  0.002325  0.002989  0.010853 -0.000914  0.017776  0.011963   

                SGLN       YCA      IPLT      SILG      REMX      INRG  \
Date                                                                     
2025-01-06 -0.006006  0.012356 -0.010022 -0.0

/Users/alexwebb/laptop_coding/risk_matrix/functions1.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):
/Users/alexwebb/laptop_coding/risk_matrix/functions1.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):
/Users/alexwebb/laptop_coding/risk_matrix/functions1.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):
/Users/alexwebb/laptop_coding/risk_matrix/functions1.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(me

In [ ]:



trc = (weights * mrc).abs()
share = trc / trc.sum()  
# print('share', share)        # risk share by asset
desired_dd = 0.04
b = desired_dd * share 
# print('b',b)    

# STOP TERMS
abs_weigths = np.abs(weights) + 1e-18
stop_pct_L1 = b / abs_weigths

# BLEND AND ENFORCE L1 CAP
lam = 0.3
stop_pct = (1 - lam) * 

